## Model Loading

In [ ]:
import torch
from pathlib import Path
import sys

# 루트 경로 추가 (Model, dataset 로딩 위해)
root_dir = Path('/home/eungyeop/LLM/tabular/ProtoLLM_entropic20251217')
sys.path.append(str(root_dir))

from models.TabularFLM_S_ import Model
from dataset.data_dataloaders import prepare_embedding_dataloaders
from utils.util import fix_seed
from ot.batch import solve_gromov_batch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#ckpt_path = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Medicaldataset+Cardiovascular_Disease_Dataset+Heart_disease_statlog+Erbil_Cardiovascular_Health_Dataset+cardio_SAheart+heart_failure_clinical_records/Pre/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_struct_hidden_dim-192_fgw_alpha-1.0_alpha-0.5_vq_beta-0.3_kl_gamma-2.0_tau-0.5_target_data-heart_entropic_reg-0.01_description-EXP3_REG_ALPHA05_FGW1/50/20260318_140439/best_joint.pt"
ckpt_path = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Medicaldataset+Cardiovascular_Disease_Dataset+Heart_disease_statlog+Erbil_Cardiovascular_Health_Dataset+cardio_SAheart+heart_failure_clinical_records_till_20260317/Pre/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_struct_hidden_dim-192_fgw_alpha-0.3_alpha-0.7_vq_beta-0.3_kl_gamma-2.0_tau-0.5_target_data-heart_entropic_reg-0.01_description-XTFORMER_EVALUATION100/50/20260225_221853/best_joint.pt"
target_rr_ckpt_path = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Medicaldataset+Cardiovascular_Disease_Dataset+Heart_disease_statlog+Erbil_Cardiovascular_Health_Dataset+cardio_SAheart+heart_failure_clinical_records/Few/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_struct_hidden_dim-192_fgw_alpha-0.3_alpha-0.7_vq_beta-0.3_kl_gamma-2.0_tau-0.5_target_data-heart_entropic_reg-0.01_description-XTFORMER_EVALUATION100/50/20260225_221853/Embed:carte_Edge:mlp_A:gat_v1_S:50_20260225_221853.pt"

ckpt = torch.load(ckpt_path, map_location=device)
args = ckpt['args']

model = Model(
    args,
    args.input_dim,
    args.hidden_dim,
    args.output_dim,
    args.dropout_rate,
    args.llm_model,
    experiment_id="viz",
    mode="viz"
).to(device)

sd = ckpt['model_state_dict']
missing, unexpected = model.load_state_dict(sd, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

model.eval()

## Source Loading

In [ ]:
args2 = args
fix_seed(args2.random_seed)

sources = args2.source_data if isinstance(args2.source_data, (list, tuple)) else [args2.source_data]

loaders = {}
for src in sources:
    print(f"Loading dataloader for source: {src}")
    res = prepare_embedding_dataloaders(args2, src, is_source=True)
    tr, va, te = res['loaders']

    from torch.utils.data import ConcatDataset, DataLoader
    datasets = [l.dataset for l in [tr, va, te] if l is not None]
    ds = ConcatDataset(datasets)

    loaders[src] = DataLoader(ds, batch_size=32, shuffle=False)

## Helpers: load tabular, one-hot basis risk, TP/TN extraction

In [ ]:
import numpy as np
import pandas as pd
import torch.nn.functional as F

base_table_path = "/storage/personal/eungyeop/dataset/table/origin_table"

dataset_and_class = {
    "heart": ['target_binary', ['no','yes']],
    "Cardiovascular_Disease_Dataset": ['target_binary', ['no', 'yes']],
    "Medicaldataset": ['target_binary', ['no', 'yes']],
    "Heart_disease_statlog": ['target_binary', ['no','yes']],
    "Erbil_Cardiovascular_Health_Dataset": ['target_binary', ['no','yes']],
    "cardio_SAheart": ['target_binary', ['no','yes']],
    "heart_failure_clinical_records": ['target_binary', ['no','yes']],
}

label_col_map = {
    "Cardiovascular_Disease_Dataset": "target",
    "Medicaldataset": "Result",
    "heart1": "output",
}

def load_original_tabular(dataset_name):
    csv_path = f"{base_table_path}/{dataset_name}.csv"
    df = pd.read_csv(csv_path)
    class_info = dataset_and_class.get(dataset_name)
    if class_info is None:
        for col in ['target_binary', 'target', 'Result', 'output', 'target_multiclass']:
            if col in df.columns:
                y = df[col]; X = df.drop(col, axis=1)
                return X.reset_index(drop=True), y.reset_index(drop=True)
        raise ValueError(f"Cannot find label column for {dataset_name}")
    class_name = class_info[0]
    if dataset_name in label_col_map:
        orig_col = label_col_map[dataset_name]
        if orig_col in df.columns:
            df[class_name] = df[orig_col]; df = df.drop(orig_col, axis=1)
    if class_name in df.columns:
        X = df.drop(class_name, axis=1); y = df[class_name]
    else:
        raise ValueError(f"Label column '{class_name}' not found")
    return X.reset_index(drop=True), y.reset_index(drop=True)


@torch.no_grad()
def compute_onehot_basis_risk(model, device):
    model.eval()
    lcg_feat, lcg_struct = model.latent_graph()
    M = lcg_feat.shape[0]
    expert_outputs = model.gnn_experts(lcg_feat.unsqueeze(0), lcg_struct.unsqueeze(0))

    onehot_probs = []
    for k in range(M):
        pi_onehot = torch.zeros(1, M, device=device)
        pi_onehot[0, k] = 1.0
        weighted = (pi_onehot.unsqueeze(-1) * expert_outputs).sum(dim=1)
        current_mode = getattr(model, 'mode', 'Full')
        logit = model.ghead2(weighted) if current_mode == 'Few' else model.ghead(weighted)
        onehot_probs.append(torch.sigmoid(logit).item())

    print("One-Hot Basis Risk:")
    for k in range(M):
        label = "HIGH" if onehot_probs[k] > 0.5 else "LOW"
        print(f"  B{k}: {onehot_probs[k]:.4f} → {label}")
    return np.array(onehot_probs)


@torch.no_grad()
def extract_tp_tn_with_tabular(model, loader, device, dataset_name):
    model.eval()
    X_orig, y_orig = load_original_tabular(dataset_name)
    all_pi, all_y, all_prob, all_sidx = [], [], [], []

    for batch in loader:
        batch_t = {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}
        out = model.predict(batch_t, return_all=True)
        global_pred = out[0] if isinstance(out, tuple) else out
        pi = model.graph_quantizer.last_pi
        prob = torch.sigmoid(global_pred).squeeze(-1)
        all_pi.append(pi.cpu().numpy())
        all_y.append(batch_t['y'].cpu().view(-1).numpy())
        all_prob.append(prob.cpu().view(-1).numpy())
        if 's_idx' in batch_t:
            sidx = batch_t['s_idx']
            if isinstance(sidx, torch.Tensor): all_sidx.append(sidx.cpu().view(-1).numpy())
            else: all_sidx.append(np.array([sidx] * pi.shape[0]))
        else:
            start = sum(len(s) for s in all_sidx)
            all_sidx.append(np.arange(start, start + pi.shape[0]))

    pis = np.concatenate(all_pi); ys = np.concatenate(all_y).astype(int)
    probs = np.concatenate(all_prob); sidxs = np.concatenate(all_sidx).astype(int)
    preds = (probs >= 0.5).astype(int)
    valid = sidxs < len(X_orig)
    if not valid.all():
        pis, ys, probs, sidxs, preds = pis[valid], ys[valid], probs[valid], sidxs[valid], preds[valid]

    tp = (ys == 1) & (preds == 1); tn = (ys == 0) & (preds == 0)
    fp = (ys == 0) & (preds == 1); fn = (ys == 1) & (preds == 0)
    X_mapped = X_orig.iloc[sidxs].reset_index(drop=True)

    print(f"[{dataset_name}] N={len(ys)}, TP={tp.sum()}, TN={tn.sum()}, FP={fp.sum()}, FN={fn.sum()}")
    return {'dataset_name': dataset_name, 'X': X_mapped, 'pis': pis, 'ys': ys,
            'probs': probs, 'preds': preds, 'argmax': pis.argmax(axis=1),
            'tp_mask': tp, 'tn_mask': tn, 'fp_mask': fp, 'fn_mask': fn}

## Build `onehot_probs` and `all_results`

In [ ]:
onehot_probs = compute_onehot_basis_risk(model, device)

all_results = {}
for src_name in sources:
    all_results[src_name] = extract_tp_tn_with_tabular(model, loaders[src_name], device, src_name)

## Radar Chart: Basis-Level TP Feature Profile

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

EXCLUDE_FEATURES = {
    'patientid', 'patient_id', 'id', 'ID',
    'time',
}


def plot_radar_basis_comparison(result, onehot_probs, max_features=8):
    """
    한 source 내에서 고위험 basis들의 TP feature profile을
    radar chart로 겹쳐서 비교
    """
    src_name = result['dataset_name']
    X = result['X']
    argmax = result['argmax']
    tp_mask = result['tp_mask']
    ys = result['ys']
    M = result['pis'].shape[1]

    short_name = src_name.replace('_Dataset', '').replace('_clinical_records', '')

    num_cols = [c for c in X.select_dtypes(include=[np.number]).columns
                if c.lower() not in {f.lower() for f in EXCLUDE_FEATURES}][:max_features]

    if len(num_cols) < 3:
        print(f"[{src_name}] Not enough features for radar chart")
        return

    X_num = X[num_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
    scaler = StandardScaler()
    scaler.fit(X_num)

    high_risk_bases = [k for k in range(M) if onehot_probs[k] > 0.5]
    active_bases = []
    for k in high_risk_bases:
        k_tp = (argmax == k) & tp_mask
        if k_tp.sum() >= 3:
            active_bases.append(k)

    print(f"[{src_name}] high_risk={high_risk_bases}, active_bases={active_bases}, num_cols={len(num_cols)}")
    if len(active_bases) < 1:
        print(f"[{src_name}] Not enough active high-risk bases for radar")
        return

    low_risk_bases = [k for k in range(M) if onehot_probs[k] <= 0.5]
    for k in sorted(low_risk_bases, key=lambda k: onehot_probs[k]):
        k_tn = (argmax == k) & (ys == 0)
        if k_tn.sum() >= 10:
            active_bases.insert(0, k)
            break

    n_features = len(num_cols)
    angles = np.linspace(0, 2 * np.pi, n_features, endpoint=False).tolist()
    angles += angles[:1]

    basis_colors = {
        'low': '#4393c3',
        'B0': '#377eb8',
        'B1': '#e41a1c',
        'B2': '#ff7f00',
        'B3': '#4daf4a',
        'B4': '#67a9cf',
        'B5': '#984ea3',
        'B6': '#f781bf',
        'B7': '#a65628',
    }

    fig, ax = plt.subplots(1, 1, figsize=(10, 10), subplot_kw=dict(polar=True))

    for k in active_bases:
        is_high = onehot_probs[k] > 0.5

        if is_high:
            k_mask = (argmax == k) & tp_mask
            label_prefix = "TP"
        else:
            k_mask = (argmax == k) & (ys == 0)
            label_prefix = "TN (ref)"

        n = k_mask.sum()
        if n < 3:
            continue

        vals = X_num.loc[k_mask].values
        z_median = np.median(scaler.transform(vals), axis=0).tolist()
        z_median += z_median[:1]

        color = basis_colors.get(f'B{k}', '#999999') if is_high else basis_colors['low']
        risk = "H" if is_high else "L"
        linestyle = '-' if is_high else '--'
        linewidth = 2.5 if is_high else 2.0
        alpha_fill = 0.15 if is_high else 0.05

        ax.plot(angles, z_median, color=color, linewidth=linewidth, linestyle=linestyle,
                label=f'B{k} [{risk}] pred={onehot_probs[k]:.2f} ({label_prefix} n={n})')
        ax.fill(angles, z_median, color=color, alpha=alpha_fill)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(num_cols, fontsize=9)
    ax.set_ylim(-2, 2)
    ax.set_yticks([-1, 0, 1])
    ax.set_yticklabels(['-1σ', 'median', '+1σ'], fontsize=8, color='gray')

    ax.legend(fontsize=9, loc='upper right', bbox_to_anchor=(1.35, 1.1))
    ax.set_title(f'[{short_name}] Basis-Level Risk Profile (Radar)\n'
                 f'(each line = median z-score of TP samples in that basis)',
                 fontsize=12, pad=20)

    plt.tight_layout()
    plt.show()


for src_name in sources:
    plot_radar_basis_comparison(all_results[src_name], onehot_probs)

## Target Adapted Model — Radar

In [ ]:
# Load target adapted checkpoint
target_ckpt = torch.load(target_rr_ckpt_path, map_location=device)
target_args = target_ckpt["args"] if isinstance(target_ckpt, dict) and "args" in target_ckpt else args

target_model = Model(
    target_args,
    target_args.input_dim,
    target_args.hidden_dim,
    target_args.output_dim,
    target_args.dropout_rate,
    target_args.llm_model,
    experiment_id="viz",
    mode="Few"
).to(device)

target_sd = target_ckpt["model_state_dict"] if isinstance(target_ckpt, dict) and "model_state_dict" in target_ckpt else target_ckpt
missing, unexpected = target_model.load_state_dict(target_sd, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)
target_model.eval()

In [ ]:
# Target dataloader (전체 데이터)
fix_seed(target_args.random_seed)
targets = target_args.target_data if isinstance(target_args.target_data, (list, tuple)) else [target_args.target_data]

target_loaders = {}
for tgt in targets:
    print(f"Loading dataloader for target: {tgt}")
    res = prepare_embedding_dataloaders(target_args, tgt, is_source=False)
    tr, va, te = res["loaders"]
    from torch.utils.data import ConcatDataset, DataLoader
    datasets = [l.dataset for l in [tr, va, te] if l is not None]
    ds = ConcatDataset(datasets)
    target_loaders[tgt] = DataLoader(ds, batch_size=32, shuffle=False)

In [ ]:
# Compute onehot risk + extract for target, then radar
target_onehot_probs = compute_onehot_basis_risk(target_model, device)

target_results = {}
for tgt in targets:
    target_results[tgt] = extract_tp_tn_with_tabular(target_model, target_loaders[tgt], device, tgt)

for tgt in targets:
    plot_radar_basis_comparison(target_results[tgt], target_onehot_probs)